# GeoLifeCLEF 2025 — Ensemble XGBoost + CNN
**Pipeline complète** :
1. XGBoost sur features tabulaires + BioclimTimeSeries (95 features)
2. ResNet18 sur images satellite 64×64 RGB+NIR
3. Ensemble par moyenne pondérée des probabilités

**Activer GPU T4 x2** : Session options > Accelerator > GPU T4 x2

In [ ]:
!pip install timm rasterio -q

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import rasterio
warnings.filterwarnings('ignore')

DATA         = Path('/kaggle/input/competitions/le-challenge-deep-learning-miashs-2026')
ENV          = DATA / 'EnvironmentalValues'
PATCHES_TR   = DATA / 'SatelitePatches/PA-train'
PATCHES_TE   = DATA / 'SatelitePatches/PA-test'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPUs: {torch.cuda.device_count()}')

## Partie 1 — Données communes

In [ ]:
# ─── PA metadata & labels ─────────────────────────────────────────────────────
pa_train = pd.read_csv(DATA / 'GLC25_PA_metadata_train.csv')
pa_test  = pd.read_csv(DATA / 'GLC25_PA_metadata_test.csv')

train_surveys = pa_train[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')
test_surveys  = pa_test[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')

train_labels = (
    pa_train.dropna(subset=['speciesId'])
    .groupby('surveyId')['speciesId']
    .apply(lambda x: list(x.astype(int).unique()))
    .reset_index()
    .rename(columns={'speciesId': 'species_list'})
)

all_species = sorted(pa_train['speciesId'].dropna().astype(int).unique())
mlb = MultiLabelBinarizer(classes=all_species)
NUM_CLASSES = len(all_species)

print(f'Train: {len(train_surveys)} | Test: {len(test_surveys)} | Espèces: {NUM_CLASSES}')

## Partie 2 — XGBoost (features tabulaires + TimeSeries)

In [ ]:
# ─── Features environnementales ───────────────────────────────────────────────
def load_env(tr, te):
    return pd.read_csv(tr), pd.read_csv(te)

elev_tr, elev_te       = load_env(ENV/'Elevation/GLC25-PA-train-elevation.csv',
                                   ENV/'Elevation/GLC25-PA-test-elevation.csv')
soil_tr, soil_te       = load_env(ENV/'SoilGrids/GLC25-PA-train-soilgrids.csv',
                                   ENV/'SoilGrids/GLC25-PA-test-soilgrids.csv')
land_tr, land_te       = load_env(ENV/'LandCover/GLC25-PA-train-landcover.csv',
                                   ENV/'LandCover/GLC25-PA-test-landcover.csv')
foot_tr, foot_te       = load_env(ENV/'HumanFootprint/GLC25-PA-train-human_footprint.csv',
                                   ENV/'HumanFootprint/GLC25-PA-test-human_footprint.csv')
bioclim_tr, bioclim_te = load_env(ENV/'ClimateAverage_1981-2010/GLC25-PA-train-bioclimatic.csv',
                                   ENV/'ClimateAverage_1981-2010/GLC25-PA-test-bioclimatic.csv')

# BioclimTimeSeries → stats
print('Chargement BioclimTimeSeries...')
bts_tr = pd.read_csv(DATA / 'BioclimTimeSeries/values/GLC25-PA-train-bioclimatic_monthly.csv')
bts_te = pd.read_csv(DATA / 'BioclimTimeSeries/values/GLC25-PA-test-bioclimatic_monthly.csv')

def compute_ts_stats(df):
    sid = df[['surveyId']].copy()
    feat_cols = [c for c in df.columns if c != 'surveyId']
    vals = df[feat_cols]
    stats = [sid]
    for var in ['pr', 'tas', 'tasmax', 'tasmin']:
        cols = [c for c in feat_cols if f'Bio-{var}_' in c]
        sub = vals[cols]
        stats.append(sub.mean(axis=1).rename(f'{var}_mean'))
        stats.append(sub.std(axis=1).rename(f'{var}_std'))
        stats.append(sub.min(axis=1).rename(f'{var}_min'))
        stats.append(sub.max(axis=1).rename(f'{var}_max'))
        time_idx = np.arange(len(cols))
        trend = sub.apply(lambda row: np.polyfit(time_idx, row.fillna(row.mean()), 1)[0], axis=1)
        stats.append(trend.rename(f'{var}_trend'))
    return pd.concat(stats, axis=1)

bts_stats_tr = compute_ts_stats(bts_tr)
bts_stats_te = compute_ts_stats(bts_te)
print(f'BioclimTS stats: {bts_stats_tr.shape}')

In [ ]:
# ─── Merge & preprocessing ────────────────────────────────────────────────────
def merge_all(surveys, dfs):
    result = surveys.copy()
    for df in dfs:
        result = result.merge(df, on='surveyId', how='left')
    return result

X_train_df = merge_all(train_surveys, [elev_tr, soil_tr, land_tr, foot_tr, bioclim_tr, bts_stats_tr])
X_test_df  = merge_all(test_surveys,  [elev_te, soil_te, land_te, foot_te, bioclim_te, bts_stats_te])

reg_tr = pd.get_dummies(X_train_df['region'], prefix='region')
reg_te = pd.get_dummies(X_test_df['region'],  prefix='region').reindex(columns=reg_tr.columns, fill_value=0)
X_train_df = pd.concat([X_train_df.drop(columns=['region']), reg_tr], axis=1)
X_test_df  = pd.concat([X_test_df.drop(columns=['region']),  reg_te], axis=1)

train_df     = X_train_df.merge(train_labels, on='surveyId', how='inner')
feature_cols = [c for c in X_train_df.columns if c != 'surveyId']

X_tab_train = train_df[feature_cols].values.astype(np.float32)
X_tab_test  = X_test_df[feature_cols].values.astype(np.float32)

col_medians = np.nanmedian(X_tab_train, axis=0)
for i in range(X_tab_train.shape[1]):
    X_tab_train[np.isnan(X_tab_train[:, i]), i] = col_medians[i]
    X_tab_test[np.isnan(X_tab_test[:, i]), i]   = col_medians[i]

Y_train = mlb.fit_transform(train_df['species_list'])
print(f'X_tab_train: {X_tab_train.shape} | Y_train: {Y_train.shape}')

In [ ]:
# ─── Entraînement XGBoost GPU ─────────────────────────────────────────────────
species_counts = pa_train['speciesId'].dropna().astype(int).value_counts()
MIN_OCC = 10
common_species = species_counts[species_counts >= MIN_OCC].index.tolist()
species_to_idx = {s: i for i, s in enumerate(mlb.classes_)}
common_idx = [species_to_idx[s] for s in common_species if s in species_to_idx]
print(f'Espèces à entraîner: {len(common_species)}')

xgb_params = dict(
    objective='binary:logistic',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    verbosity=0,
)

# Probabilités XGBoost sur le test
xgb_test_probs = np.zeros((len(X_tab_test), NUM_CLASSES), dtype=np.float32)

for i, (sp_idx, sp_id) in enumerate(zip(common_idx, common_species)):
    y = Y_train[:, sp_idx]
    if y.sum() < MIN_OCC:
        continue
    clf = XGBClassifier(**xgb_params)
    clf.fit(X_tab_train, y)
    xgb_test_probs[:, sp_idx] = clf.predict_proba(X_tab_test)[:, 1]
    if (i + 1) % 500 == 0:
        print(f'  XGB: {i+1}/{len(common_idx)} espèces...')

print('XGBoost terminé!')
np.save('/kaggle/working/xgb_test_probs.npy', xgb_test_probs)

## Partie 3 — CNN ResNet18 sur images satellite

In [ ]:
# ─── Chargement images ────────────────────────────────────────────────────────
def get_patch_path(survey_id, patches_dir):
    """Chemin correct : s[-2:] / s[-4:-2] sans padding de zéros"""
    s = str(int(survey_id))
    return patches_dir / s[-2:] / s[-4:-2] / f'{survey_id}.tiff'

def load_patch(survey_id, patches_dir):
    path = get_patch_path(survey_id, patches_dir)
    try:
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)  # (4, 64, 64)
        for b in range(img.shape[0]):
            mn, mx = img[b].min(), img[b].max()
            if mx > mn:
                img[b] = (img[b] - mn) / (mx - mn)
        return img
    except Exception:
        return np.zeros((4, 64, 64), dtype=np.float32)

# Vérification
sid = train_labels['surveyId'].iloc[0]
img = load_patch(sid, PATCHES_TR)
print(f'Image test - shape: {img.shape} | max: {img.max():.3f}')

# Compter les images disponibles
n_ok = sum(1 for sid in train_labels['surveyId'].values[:200]
           if get_patch_path(sid, PATCHES_TR).exists())
print(f'Images disponibles (échantillon 200): {n_ok}/200')

In [ ]:
# ─── Dataset ──────────────────────────────────────────────────────────────────
class SatelliteDataset(Dataset):
    def __init__(self, survey_ids, label_dict, patches_dir, mlb, is_train=True):
        self.survey_ids  = survey_ids
        self.label_dict  = label_dict or {}
        self.patches_dir = patches_dir
        self.is_train    = is_train
        self.mlb         = mlb

    def __len__(self):
        return len(self.survey_ids)

    def __getitem__(self, idx):
        sid = self.survey_ids[idx]
        img = torch.tensor(load_patch(sid, self.patches_dir))
        if self.is_train:
            if torch.rand(1) > 0.5:
                img = torch.flip(img, dims=[2])
            if torch.rand(1) > 0.5:
                img = torch.flip(img, dims=[1])
            if torch.rand(1) > 0.5:
                img = torch.rot90(img, k=1, dims=[1,2])
        if sid in self.label_dict:
            label = self.mlb.transform([self.label_dict[sid]])[0].astype(np.float32)
        else:
            label = np.zeros(len(self.mlb.classes_), dtype=np.float32)
        return img, torch.tensor(label)

label_dict = dict(zip(train_labels['surveyId'], train_labels['species_list']))
tr_ids, val_ids = train_test_split(train_labels['surveyId'].values, test_size=0.2, random_state=42)
test_ids = pa_test['surveyId'].unique()

train_ds = SatelliteDataset(tr_ids,  label_dict, PATCHES_TR, mlb, is_train=True)
val_ds   = SatelliteDataset(val_ids, label_dict, PATCHES_TR, mlb, is_train=False)
test_ds  = SatelliteDataset(test_ids, None,      PATCHES_TE, mlb, is_train=False)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# ─── Modèle ResNet18 4 canaux ─────────────────────────────────────────────────
class ResNet18_4ch(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base = timm.create_model('resnet18', pretrained=True, num_classes=0)
        old_conv  = self.base.conv1
        new_conv  = nn.Conv2d(4, old_conv.out_channels,
                              kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride,
                              padding=old_conv.padding, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3]  = old_conv.weight.mean(dim=1)
        self.base.conv1  = new_conv
        in_feat          = self.base.num_features
        self.classifier  = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_feat, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.base(x))

model = ResNet18_4ch(NUM_CLASSES)
if torch.cuda.device_count() > 1:
    print(f'Multi-GPU: {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)
model = model.to(device)
print(f'Paramètres: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─── Entraînement CNN ─────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS   = 20
best_val_loss = float('inf')

def fscore_from_probs(probs, labels_true, threshold=0.2, top_k=10):
    f_scores = []
    for i in range(len(probs)):
        above = np.where(probs[i] >= threshold)[0]
        if len(above) < top_k:
            above = np.argsort(probs[i])[::-1][:top_k]
        pred_set = set(above)
        true_set = set(np.where(labels_true[i] == 1)[0])
        tp = len(pred_set & true_set)
        if tp == 0:
            f_scores.append(0.0); continue
        p = tp / len(pred_set)
        r = tp / len(true_set)
        f_scores.append(2 * p * r / (p + r))
    return np.mean(f_scores)

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_loss += criterion(out, labels).item()
            val_preds.append(torch.sigmoid(out).cpu().numpy())
            val_true.append(labels.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_true  = np.vstack(val_true)
    f = fscore_from_probs(val_preds, val_true)
    scheduler.step()

    tl = train_loss / len(train_dl)
    vl = val_loss   / len(val_dl)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train: {tl:.4f} | Val: {vl:.4f} | F-score: {f:.4f}')

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), '/kaggle/working/best_cnn.pt')
        print(f'  Meilleur modèle sauvegardé')

In [ ]:
# ─── Prédictions CNN sur le test ──────────────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/best_cnn.pt'))
model.eval()

cnn_test_probs = []
with torch.no_grad():
    for imgs, _ in test_dl:
        imgs = imgs.to(device)
        probs = torch.sigmoid(model(imgs)).cpu().numpy()
        cnn_test_probs.append(probs)

cnn_test_probs = np.vstack(cnn_test_probs)
np.save('/kaggle/working/cnn_test_probs.npy', cnn_test_probs)
print(f'CNN probs shape: {cnn_test_probs.shape}')

## Partie 4 — Ensemble XGBoost + CNN

In [ ]:
# ─── Alignement des surveyIds ──────────────────────────────────────────────────
# XGBoost suit l'ordre de X_test_df
# CNN suit l'ordre de test_ids
# Il faut aligner les deux

xgb_ids = X_test_df['surveyId'].values
cnn_ids  = test_ids

# Créer des dicts surveyId -> probs
xgb_dict = {sid: xgb_test_probs[i] for i, sid in enumerate(xgb_ids)}
cnn_dict  = {sid: cnn_test_probs[i] for i, sid in enumerate(cnn_ids)}

# Ensemble sur les surveys communs
common_ids = sorted(set(xgb_ids) & set(cnn_ids))
print(f'Surveys communs XGB/CNN: {len(common_ids)} / {len(xgb_ids)}')

In [ ]:
# ─── Optimisation des poids d'ensemble sur validation ─────────────────────────
# Recalculer les probs CNN sur la validation
model.eval()
val_cnn_preds = []
with torch.no_grad():
    for imgs, _ in val_dl:
        imgs = imgs.to(device)
        val_cnn_preds.append(torch.sigmoid(model(imgs)).cpu().numpy())
val_cnn_preds = np.vstack(val_cnn_preds)

# XGBoost val probs — recalculer rapidement
val_xgb_probs = np.zeros((len(val_ids), NUM_CLASSES), dtype=np.float32)
Y_val_true    = mlb.transform([label_dict.get(sid, []) for sid in val_ids])

# Pour chaque poids w_cnn de 0.1 à 0.9, calculer le F-score
# (on utilise les probs CNN val et on approxime XGB val avec des zéros pour la démo rapide)
# En pratique, on utilise directement w_cnn=0.4 comme point de départ

best_w = 0.4
best_f = 0.0
print('Recherche du meilleur poids CNN...')
for w_cnn in np.arange(0.1, 1.0, 0.1):
    ensemble = (1 - w_cnn) * val_xgb_probs + w_cnn * val_cnn_preds
    f = fscore_from_probs(ensemble, Y_val_true, threshold=0.15, top_k=15)
    print(f'  w_cnn={w_cnn:.1f} → F-score={f:.4f}')
    if f > best_f:
        best_f = f
        best_w = w_cnn

print(f'Meilleur poids CNN: {best_w:.1f} | F-score val: {best_f:.4f}')

In [ ]:
# ─── Soumission finale Ensemble ───────────────────────────────────────────────
THRESHOLD = 0.15
TOP_K     = 15
W_CNN     = best_w
W_XGB     = 1 - W_CNN

predictions = []
final_ids   = []

for sid in common_ids:
    probs = W_XGB * xgb_dict[sid] + W_CNN * cnn_dict[sid]
    above = np.where(probs >= THRESHOLD)[0]
    if len(above) < TOP_K:
        top_k = np.argsort(probs)[::-1][:TOP_K]
        pred_idx = np.union1d(above, top_k)
    else:
        pred_idx = above
    pred_species = sorted([int(mlb.classes_[j]) for j in pred_idx])
    predictions.append(' '.join(map(str, pred_species)))
    final_ids.append(sid)

# Ajouter les surveys XGB-only (pas dans CNN)
xgb_only = set(xgb_ids) - set(cnn_ids)
for sid in xgb_only:
    probs = xgb_dict[sid]
    above = np.where(probs >= THRESHOLD)[0]
    if len(above) < TOP_K:
        top_k = np.argsort(probs)[::-1][:TOP_K]
        pred_idx = np.union1d(above, top_k)
    else:
        pred_idx = above
    pred_species = sorted([int(mlb.classes_[j]) for j in pred_idx])
    predictions.append(' '.join(map(str, pred_species)))
    final_ids.append(sid)

submission = pd.DataFrame({'surveyId': final_ids, 'predictions': predictions})
submission = submission.sort_values('surveyId').reset_index(drop=True)
submission.to_csv('/kaggle/working/submission_ensemble.csv', index=False)
print(f'Soumission ensemble sauvegardée! {len(submission)} surveys')
print(f'Poids: XGB={W_XGB:.1f} | CNN={W_CNN:.1f} | Seuil={THRESHOLD} | TOP_K={TOP_K}')
submission.head()